In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q -U transformers datasets accelerate scikit-learn pandas pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 

In [1]:
%%writefile centralized_tinybert_imdb.py
import os
import re
import gc
import json
import time
import math
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    set_seed,
)
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

# =========================================================
# CONFIG
# =========================================================
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
DATASET_NAME = "imdb"
SETTING_NAME = "centralized_tinybert_full_finetuning"

OUTPUT_DIR = "/content/drive/MyDrive/centralized_tinybert_imdb"
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, "final_model")

HISTORY_JSON = os.path.join(OUTPUT_DIR, "history.json")
HISTORY_CSV = os.path.join(OUTPUT_DIR, "history.csv")
RUN_SUMMARY_JSON = os.path.join(OUTPUT_DIR, "run_summary.json")
CONFIG_JSON = os.path.join(OUTPUT_DIR, "config.json")

SEED = 42
MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
LR = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1

EARLY_STOPPING_PATIENCE = 3
MONITOR_METRIC = "accuracy"

# chỉ là trần an toàn; thực tế dừng bằng early stopping
MAX_EPOCHS = 1000000

NUM_WORKERS = 0
USE_FP16 = True

# =========================================================
# UTILS
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def clear_and_make_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_csv(rows, path):
    if len(rows) == 0:
        pd.DataFrame([]).to_csv(path, index=False)
    else:
        pd.DataFrame(rows).to_csv(path, index=False)

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def tokenize_function(batch, tokenizer, max_length):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
    )

def list_checkpoints(checkpoints_dir):
    ckpts = []
    if not os.path.exists(checkpoints_dir):
        return ckpts
    for name in os.listdir(checkpoints_dir):
        full = os.path.join(checkpoints_dir, name)
        if os.path.isdir(full) and name.startswith("checkpoint_epoch_"):
            m = re.search(r"checkpoint_epoch_(\d+)", name)
            if m:
                ckpts.append((int(m.group(1)), full))
    ckpts.sort(key=lambda x: x[0])
    return ckpts

def latest_checkpoint_path(checkpoints_dir):
    ckpts = list_checkpoints(checkpoints_dir)
    return ckpts[-1][1] if len(ckpts) > 0 else None

def keep_latest_k_checkpoints(checkpoints_dir, keep_k=2):
    ckpts = list_checkpoints(checkpoints_dir)
    while len(ckpts) > keep_k:
        _, old_path = ckpts.pop(0)
        shutil.rmtree(old_path, ignore_errors=True)

def improved(current, best):
    if best is None:
        return True
    return current > best

# =========================================================
# MODEL
# =========================================================
def build_model():
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        ignore_mismatched_sizes=True,
    )
    model.config.problem_type = "single_label_classification"
    return model

# =========================================================
# EVAL
# =========================================================
@torch.no_grad()
def evaluate(model, dataloader, device, use_fp16=False):
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_preds = []
    all_labels = []

    amp_enabled = bool(use_fp16 and device.type == "cuda")

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        if amp_enabled:
            with torch.amp.autocast("cuda"):
                outputs = model(**batch)
                loss = outputs.loss
                logits = outputs.logits
        else:
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

        bs = batch["labels"].size(0)
        total_loss += loss.item() * bs
        total_count += bs

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(total_count, 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return {
        "eval_loss": float(avg_loss),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
    }

# =========================================================
# SAVE / LOAD
# =========================================================
def save_model_bundle(save_dir, model, tokenizer):
    clear_and_make_dir(save_dir)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    metadata = {
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "setting": SETTING_NAME,
        "max_length": MAX_LENGTH,
    }
    save_json(metadata, os.path.join(save_dir, "model_metadata.json"))

def save_checkpoint(
    epoch_idx,
    model,
    tokenizer,
    optimizer,
    scheduler,
    scaler,
    best_metric_so_far,
    patience_counter,
    history,
):
    ckpt_dir = os.path.join(CHECKPOINTS_DIR, f"checkpoint_epoch_{epoch_idx:06d}")
    ensure_dir(ckpt_dir)

    model.save_pretrained(os.path.join(ckpt_dir, "model"))
    tokenizer.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))

    state = {
        "epoch": int(epoch_idx),
        "best_metric_so_far": None if best_metric_so_far is None else float(best_metric_so_far),
        "patience_counter": int(patience_counter),
        "history": history,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": None if scaler is None else scaler.state_dict(),
        "config": {
            "model_name": MODEL_NAME,
            "dataset_name": DATASET_NAME,
            "setting": SETTING_NAME,
            "output_dir": OUTPUT_DIR,
            "checkpoints_dir": CHECKPOINTS_DIR,
            "best_model_dir": BEST_MODEL_DIR,
            "final_model_dir": FINAL_MODEL_DIR,
            "history_json": HISTORY_JSON,
            "history_csv": HISTORY_CSV,
            "run_summary_json": RUN_SUMMARY_JSON,
            "config_json": CONFIG_JSON,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "max_length": MAX_LENGTH,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "seed": SEED,
        }
    }
    torch.save(state, os.path.join(ckpt_dir, "training_state.pt"))

    with open(os.path.join(CHECKPOINTS_DIR, "latest_checkpoint.txt"), "w", encoding="utf-8") as f:
        f.write(ckpt_dir)

    keep_latest_k_checkpoints(CHECKPOINTS_DIR, keep_k=2)
    return ckpt_dir

def load_latest_checkpoint(device):
    ckpt_path = latest_checkpoint_path(CHECKPOINTS_DIR)
    if ckpt_path is None:
        return None

    model_dir = os.path.join(ckpt_path, "model")
    tokenizer_dir = os.path.join(ckpt_path, "tokenizer")
    state_path = os.path.join(ckpt_path, "training_state.pt")

    if not (os.path.exists(model_dir) and os.path.exists(tokenizer_dir) and os.path.exists(state_path)):
        return None

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)

    state = torch.load(state_path, map_location=device)

    return {
        "ckpt_path": ckpt_path,
        "model": model,
        "tokenizer": tokenizer,
        "start_epoch": int(state["epoch"]) + 1,
        "best_metric_so_far": state["best_metric_so_far"],
        "patience_counter": int(state["patience_counter"]),
        "history": state["history"],
        "optimizer_state_dict": state["optimizer_state_dict"],
        "scheduler_state_dict": state["scheduler_state_dict"],
        "scaler_state_dict": state["scaler_state_dict"],
    }

# =========================================================
# MAIN
# =========================================================
def main():
    ensure_dir(OUTPUT_DIR)
    ensure_dir(CHECKPOINTS_DIR)
    ensure_dir(BEST_MODEL_DIR)
    ensure_dir(FINAL_MODEL_DIR)

    config_dump = {
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "setting": SETTING_NAME,
        "output_dir": OUTPUT_DIR,
        "checkpoints_dir": CHECKPOINTS_DIR,
        "best_model_dir": BEST_MODEL_DIR,
        "final_model_dir": FINAL_MODEL_DIR,
        "history_json": HISTORY_JSON,
        "history_csv": HISTORY_CSV,
        "run_summary_json": RUN_SUMMARY_JSON,
        "config_json": CONFIG_JSON,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "monitor_metric": MONITOR_METRIC,
        "seed": SEED,
    }
    save_json(config_dump, CONFIG_JSON)

    set_seed(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    device = get_device()

    print("=" * 120)
    print("Centralized Training: TinyBERT (full fine-tuning) on IMDB")
    print(f"Device: {device}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Checkpoints directory: {CHECKPOINTS_DIR}")
    print(f"Best model directory: {BEST_MODEL_DIR}")
    print(f"Final model directory: {FINAL_MODEL_DIR}")
    print("=" * 120)

    resume_info = load_latest_checkpoint(device=device)

    if resume_info is not None:
        print(f"[Resume] Found latest checkpoint: {resume_info['ckpt_path']}")
        tokenizer = resume_info["tokenizer"]
        model = resume_info["model"]
        start_epoch = resume_info["start_epoch"]
        best_metric_so_far = resume_info["best_metric_so_far"]
        patience_counter = resume_info["patience_counter"]
        history = resume_info["history"]
        print(f"[Resume] Continuing from epoch {start_epoch}")
    else:
        print("[Start] No checkpoint found. Starting from scratch.")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
        model = build_model().to(device)
        start_epoch = 1
        best_metric_so_far = None
        patience_counter = 0
        history = []

    raw_datasets = load_dataset(DATASET_NAME)

    tokenized_train = raw_datasets["train"].map(
        lambda batch: tokenize_function(batch, tokenizer, MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )
    tokenized_test = raw_datasets["test"].map(
        lambda batch: tokenize_function(batch, tokenizer, MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )

    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_test = tokenized_test.rename_column("label", "labels")

    tokenized_train.set_format(type="torch")
    tokenized_test.set_format(type="torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_loader = DataLoader(
        tokenized_train,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=data_collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    eval_loader = DataLoader(
        tokenized_test,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    trainable_params, total_params = count_parameters(model)

    total_steps_per_epoch = max(len(train_loader), 1)
    total_training_steps = max(MAX_EPOCHS * total_steps_per_epoch, 1)
    warmup_steps = int(WARMUP_RATIO * total_steps_per_epoch)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

    use_amp = bool(USE_FP16 and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if device.type == "cuda" else None

    if resume_info is not None:
        optimizer.load_state_dict(resume_info["optimizer_state_dict"])
        scheduler.load_state_dict(resume_info["scheduler_state_dict"])
        if scaler is not None and resume_info["scaler_state_dict"] is not None:
            scaler.load_state_dict(resume_info["scaler_state_dict"])

    print(f"Train samples: {len(tokenized_train)}")
    print(f"Validation samples: {len(tokenized_test)}")
    print(f"Model: {MODEL_NAME}")
    print(f"Trainable params: {trainable_params}")
    print(f"Total params: {total_params}")

    stopped_early = False
    epoch_idx = start_epoch

    while True:
        print("\n" + "=" * 40 + f" Epoch {epoch_idx} " + "=" * 40, flush=True)
        epoch_start_time = time.time()
        model.train()

        running_loss = 0.0
        total_seen = 0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast("cuda"):
                    outputs = model(**batch)
                    loss = outputs.loss
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()

            bs = batch["labels"].size(0)
            running_loss += loss.item() * bs
            total_seen += bs

        train_loss = running_loss / max(total_seen, 1)

        eval_result = evaluate(model, eval_loader, device=device, use_fp16=USE_FP16)
        epoch_time = time.time() - epoch_start_time

        current_metric = float(eval_result[MONITOR_METRIC])
        is_new_best = improved(current_metric, best_metric_so_far)

        if is_new_best:
            best_metric_so_far = current_metric
            patience_counter = 0
            save_model_bundle(BEST_MODEL_DIR, model, tokenizer)
        else:
            patience_counter += 1

        epoch_log = {
            "epoch": int(epoch_idx),
            "train_loss": float(train_loss),
            "eval_loss": float(eval_result["eval_loss"]),
            "accuracy": float(eval_result["accuracy"]),
            "macro_f1": float(eval_result["macro_f1"]),
            "time_per_epoch": float(epoch_time),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": str(MODEL_NAME),
            "dataset_name": str(DATASET_NAME),
            "setting": str(SETTING_NAME),
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }
        history.append(epoch_log)

        save_json(history, HISTORY_JSON)
        save_csv(history, HISTORY_CSV)

        ckpt_dir = save_checkpoint(
            epoch_idx=epoch_idx,
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            best_metric_so_far=best_metric_so_far,
            patience_counter=patience_counter,
            history=history,
        )

        print("-" * 120)
        print("EPOCH LOG:")
        print(json.dumps(epoch_log, indent=2))
        print(f"Checkpoint saved: {ckpt_dir}")
        print("-" * 120)

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"[Early Stopping] No validation {MONITOR_METRIC} improvement for {EARLY_STOPPING_PATIENCE} consecutive epochs.")
            stopped_early = True
            break

        epoch_idx += 1

        if epoch_idx > MAX_EPOCHS:
            print("[Stop] Reached MAX_EPOCHS safety ceiling.")
            break

    save_model_bundle(FINAL_MODEL_DIR, model, tokenizer)
    save_json(history, HISTORY_JSON)
    save_csv(history, HISTORY_CSV)

    run_summary = {
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "setting": SETTING_NAME,
        "output_dir": OUTPUT_DIR,
        "checkpoints_dir": CHECKPOINTS_DIR,
        "best_model_dir": BEST_MODEL_DIR,
        "final_model_dir": FINAL_MODEL_DIR,
        "history_json": HISTORY_JSON,
        "history_csv": HISTORY_CSV,
        "num_epochs_completed": int(len(history)),
        "best_metric_so_far": None if best_metric_so_far is None else float(best_metric_so_far),
        "stopped_early": bool(stopped_early),
        "latest_checkpoint": latest_checkpoint_path(CHECKPOINTS_DIR),
    }
    save_json(run_summary, RUN_SUMMARY_JSON)

    expected_cols = [
        "epoch",
        "train_loss",
        "eval_loss",
        "accuracy",
        "macro_f1",
        "time_per_epoch",
        "trainable_params",
        "total_params",
        "model_name",
        "dataset_name",
        "setting",
        "best_metric_so_far",
        "patience_counter",
        "is_new_best",
    ]

    history_df = pd.DataFrame(history)
    missing_cols = [c for c in expected_cols if c not in history_df.columns]

    required_paths = [
        OUTPUT_DIR,
        CHECKPOINTS_DIR,
        BEST_MODEL_DIR,
        FINAL_MODEL_DIR,
        HISTORY_JSON,
        HISTORY_CSV,
        RUN_SUMMARY_JSON,
        CONFIG_JSON,
    ]
    missing_files = [p for p in required_paths if not os.path.exists(p)]

    print("=" * 120)
    print("Training completed.")
    print(f"Output directory: {OUTPUT_DIR}")
    print("Generated files/folders:")
    print(f" - {HISTORY_JSON}")
    print(f" - {HISTORY_CSV}")
    print(f" - {RUN_SUMMARY_JSON}")
    print(f" - {CONFIG_JSON}")
    print(f" - {CHECKPOINTS_DIR}/latest_checkpoint.txt")
    print(f" - {CHECKPOINTS_DIR}/checkpoint_epoch_xxxxxx/")
    print(f" - {BEST_MODEL_DIR}/")
    print(f" - {FINAL_MODEL_DIR}/")
    print("-" * 120)
    print("[SELF-CHECK]")
    print(f"Missing history columns: {missing_cols}")
    print(f"Missing files/folders: {missing_files}")
    print("=" * 120)

    if len(missing_cols) == 0 and len(missing_files) == 0:
        print("[SELF-CHECK PASSED] All required columns and output files are present.")
    else:
        raise ValueError("SELF-CHECK FAILED: missing required columns or files.")

if __name__ == "__main__":
    main()

Writing centralized_tinybert_imdb.py


In [2]:
!python centralized_tinybert_imdb.py

Centralized Training: TinyBERT (full fine-tuning) on IMDB
Device: cuda
Output directory: /content/drive/MyDrive/centralized_tinybert_imdb
Checkpoints directory: /content/drive/MyDrive/centralized_tinybert_imdb/checkpoints
Best model directory: /content/drive/MyDrive/centralized_tinybert_imdb/best_model
Final model directory: /content/drive/MyDrive/centralized_tinybert_imdb/final_model
[Start] No checkpoint found. Starting from scratch.
config.json: 100% 409/409 [00:00<00:00, 2.38MB/s]
vocab.txt: 232kB [00:00, 67.7MB/s]
pytorch_model.bin: 100% 62.7M/62.7M [00:02<00:00, 23.0MB/s]
Loading weights: 100% 71/71 [00:00<00:00, 4381.28it/s, Materializing param=bert.pooler.dense.weight]
README.md: 7.81kB [00:00, 21.0MB/s]
model.safetensors: 100% 62.7M/62.7M [00:00<00:00, 76.8MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:00<00:00, 51.2MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 33.5MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M